## PropertyLens RAG — Notebook C: Evaluation (v1)

**Purpose:** evaluate the retrieval and generation quality of the RAG pipeline built in Notebooks A + B.

**Assumes:** Notebook A (`04_propertylens_build_index.ipynb`) has populated the Pinecone index and `bm25_encoder_v3.pkl` exists on disk.

**Three evaluation metrics — in order of importance for PropertyLens:**

| # | Metric | What it measures | Why it matters here |
|---|--------|-----------------|--------------------|
| 1 | **Hit Rate @ 5** | Does at least one relevant chunk appear in top-5? | Validates the hybrid BM25+cosine+RRF+reranker stack |
| 2 | **Faithfulness** | Are answer claims grounded in retrieved context? | Prevents hallucinated prices influencing buyer/seller decisions |
| 3 | **Answer relevance** | Does the answer address the question? | Catches Gemma dodging questions or answering the wrong thing |

**No external APIs required.** All three metrics use Gemma 3 locally via Ollama as the judge.

**Output:** a summary table printed at the end with per-query scores and aggregate means.


### Install dependencies

Same as Notebook B — no new packages needed.


In [ ]:
!pip install -q pinecone pinecone-text sentence-transformers transformers torch \
               ollama pandas numpy python-dotenv psutil

### Configuration

Copy of Notebook B config — must match exactly so we connect to the same index with the same encoders.


In [ ]:
from __future__ import annotations
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
PINECONE_INDEX   = "propertylens-rag"
NS_TRANSACTIONS  = "transactions"
NS_AMENITIES     = "amenities"
NS_XAI           = "xai"
NS_TRENDS        = "trends"

assert PINECONE_API_KEY, "Missing PINECONE_API_KEY in .env"

DENSE_MODEL_NAME     = "BAAI/bge-m3"
RERANKER_MODEL       = "BAAI/bge-reranker-v2-m3"
CROSS_ENCODER_DEVICE = "cpu"

TOP_K_RETRIEVAL = 50
TOP_K_RERANK    = 10
TOP_K_MMR       = 5
TOP_K_FINAL     = 5
MMR_LAMBDA      = 0.7
RRF_K           = 60
N_SUBQUERIES    = 3

SOURCE_WEIGHTS: dict[str, float] = {
    "transaction": 1.0,
    "amenity":     2.5,
    "trend":       1.0,
    "xai":         2.5,
}

OLLAMA_MODEL    = "gemma3"
OLLAMA_BASE_URL = "http://localhost:11434"

def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        if (p / "data").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError("Cannot find repo root.")

REPO_ROOT       = find_repo_root()
BM25_CACHE_PATH = REPO_ROOT / "notebooks" / "05_chatbot" / "bm25_encoder_v3.pkl"

print("Config loaded.")
print(f"  Pinecone index   : {PINECONE_INDEX}")
print(f"  BM25 cache       : {BM25_CACHE_PATH}")
print(f"  Source weights   : {SOURCE_WEIGHTS}")

### Connect to Pinecone + load encoders

Same setup as Notebook B. Cross-encoder pinned to CPU.


In [ ]:
from __future__ import annotations
import gc
import pickle
import psutil
import torch
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
from pinecone_text.sparse import BM25Encoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer

_PROC = psutil.Process(os.getpid())

def mem(label: str) -> None:
    gc.collect()
    rss = _PROC.memory_info().rss / (1024 * 1024)
    print(f"  [MEM] {label:<32s} RSS = {rss:8.1f} MB")


# ── Pinecone ──────────────────────────────────────────────────────────────────
pc    = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX)
stats = index.describe_index_stats()
total = stats.get("total_vector_count") if isinstance(stats, dict) else getattr(stats, "total_vector_count", 0)
print(f"Pinecone: {total:,} vectors across namespaces")
if not total:
    print("  ⚠ Index appears empty. Run Notebook A first.")

# ── Dense encoder ─────────────────────────────────────────────────────────────
dense_encoder = SentenceTransformer(DENSE_MODEL_NAME)
print(f"Dense encoder: {DENSE_MODEL_NAME}")

# ── BM25 encoder ─────────────────────────────────────────────────────────────
if not BM25_CACHE_PATH.exists():
    raise FileNotFoundError(f"BM25 cache missing: {BM25_CACHE_PATH}. Run Notebook A first.")
with BM25_CACHE_PATH.open("rb") as f:
    bm25_encoder: BM25Encoder = pickle.load(f)
print(f"BM25 encoder: loaded from cache")

# ── Cross-encoder (CPU) ───────────────────────────────────────────────────────
ce_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL)
ce_model     = AutoModelForSequenceClassification.from_pretrained(RERANKER_MODEL)
ce_model.to(CROSS_ENCODER_DEVICE)
ce_model.eval()
print(f"Cross-encoder: {RERANKER_MODEL} on {CROSS_ENCODER_DEVICE}")

mem("after all models loaded")

### Retrieval pipeline (copied from Notebook B)

Exact copy of the inference pipeline — filter extraction, namespace routing, hybrid retrieval,
weighted RRF, cross-encoder rerank, MMR, lost-in-middle reorder, and answer generation.
No changes — we evaluate the pipeline as-is.


In [ ]:
from __future__ import annotations
import json, re
from typing import Any, Optional
import numpy as np
import ollama


# ── Ollama helper ─────────────────────────────────────────────────────────────
def _set_ollama_host(base_url: str) -> None:
    os.environ["OLLAMA_HOST"] = base_url


# ── Filter extraction ─────────────────────────────────────────────────────────
def extract_filters_from_query(query: str, model: str = OLLAMA_MODEL) -> dict | None:
    """Extract town/flat_type/sale_year from free-text query via Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Extract structured fields from this Singapore HDB property query.
Return ONLY a valid JSON object with these optional keys:
  - "town": ALL CAPS HDB town e.g. "TAMPINES", "BEDOK", "SERANGOON"
  - "flat_type": one of "2 ROOM","3 ROOM","4 ROOM","5 ROOM","EXECUTIVE"
  - "sale_year": integer year if mentioned
Omit any field you are not sure about. Return {{}} if nothing is clear.
Return ONLY JSON.

Query: {query}
"""
    try:
        response = ollama.chat(model=model, messages=[{"role": "user", "content": prompt}])
        raw      = (response.get("message") or {}).get("content", "{}")
        raw      = re.sub(r"```[\w]*", "", raw).strip()
        parsed   = json.loads(raw)
        return parsed if parsed else None
    except Exception:
        return None


# ── Namespace router ──────────────────────────────────────────────────────────
def route_namespaces(query: str) -> list[str]:
    """Select namespaces to query based on keywords."""
    q  = query.lower()
    ns = [NS_TRANSACTIONS]
    if any(kw in q for kw in ["mrt", "school", "mall", "hawker", "near", "amenity", "transport"]):
        ns.append(NS_AMENITIES)
    if any(kw in q for kw in ["trend", "rising", "falling", "increase", "history", "recent", "past"]):
        ns.append(NS_TRENDS)
    if any(kw in q for kw in ["explain", "shap", "feature", "why", "reason", "driver", "factor"]):
        ns.append(NS_XAI)
    return ns


# ── Hybrid query ──────────────────────────────────────────────────────────────
def _scale_sparse(sparse: dict, scale: float) -> dict:
    return {"indices": sparse["indices"], "values": [v * scale for v in sparse["values"]]}


def _hybrid_query(
    query: str, alpha: float, top_k: int, namespace: str,
    metadata_filter: Optional[dict] = None,
) -> list[dict]:
    dense  = dense_encoder.encode(query, normalize_embeddings=True).tolist()
    dense  = (np.array(dense, dtype=np.float32) * alpha).tolist()
    sparse = bm25_encoder.encode_queries([query])[0]
    sparse = _scale_sparse(sparse, 1.0 - alpha)
    res    = index.query(
        vector=dense, sparse_vector=sparse, top_k=top_k,
        namespace=namespace, include_metadata=True, filter=metadata_filter or None,
    )
    matches = res.get("matches") if isinstance(res, dict) else getattr(res, "matches", [])
    return [
        {"id":       getattr(m, "id",       m.get("id")),
         "score":    getattr(m, "score",    m.get("score")),
         "metadata": getattr(m, "metadata", m.get("metadata", {}))}
        for m in (matches or [])
    ]


# ── Weighted RRF ──────────────────────────────────────────────────────────────
def reciprocal_rank_fusion(
    ranked_lists: list[list[dict]],
    k: int = 60,
) -> list[dict]:
    """RRF with SOURCE_WEIGHTS boosting amenity/xai chunks."""
    scores: dict[str, float] = {}
    best:   dict[str, dict]  = {}
    for lst in ranked_lists:
        for rank, r in enumerate(lst, start=1):
            rid    = str(r.get("id", ""))
            if not rid:
                continue
            source = str((r.get("metadata") or {}).get("source", "transaction"))
            weight = SOURCE_WEIGHTS.get(source, 1.0)
            scores[rid] = scores.get(rid, 0.0) + weight * (1.0 / (float(k) + float(rank)))
            if rid not in best:
                best[rid] = r
    fused = [{**best[rid], "rrf_score": sc} for rid, sc in scores.items()]
    fused.sort(key=lambda x: x.get("rrf_score", 0.0), reverse=True)
    return fused


# ── Cross-encoder rerank ──────────────────────────────────────────────────────
def rerank_cross_encoder(query: str, candidates: list[dict], top_k: int) -> list[dict]:
    if not candidates:
        return []
    texts  = [str((c.get("metadata") or {}).get("parent_text") or "") for c in candidates]
    pairs  = [(query, t) for t in texts]
    inputs = ce_tokenizer(pairs, padding=True, truncation=True, max_length=512, return_tensors="pt")
    inputs = {k: v.to(CROSS_ENCODER_DEVICE) for k, v in inputs.items()}
    with torch.no_grad():
        logits = ce_model(**inputs).logits.squeeze(-1).tolist()
    if isinstance(logits, float):
        logits = [logits]
    scored = [{**c, "ce_score": float(s)} for c, s in zip(candidates, logits)]
    scored.sort(key=lambda x: x["ce_score"], reverse=True)
    return scored[:top_k]


# ── MMR ───────────────────────────────────────────────────────────────────────
def mmr_filter(candidates: list[dict], query: str, top_k: int, lam: float = 0.7) -> list[dict]:
    if not candidates:
        return []
    texts   = [str((c.get("metadata") or {}).get("parent_text") or "") for c in candidates]
    doc_emb = np.asarray(dense_encoder.encode(texts, normalize_embeddings=True), dtype=np.float32)
    sel, rem = [], list(range(len(candidates)))
    best0 = int(np.argmax([c.get("ce_score", -1e9) for c in candidates]))
    sel.append(best0); rem.remove(best0)
    while rem and len(sel) < top_k:
        bi, bv = None, -1e18
        for i in rem:
            rel     = float(candidates[i].get("ce_score", 0.0))
            max_sim = max(float(np.dot(doc_emb[i], doc_emb[j]) /
                               max(np.linalg.norm(doc_emb[i]) * np.linalg.norm(doc_emb[j]), 1e-12))
                          for j in sel)
            score = lam * rel - (1.0 - lam) * max_sim
            if score > bv:
                bv, bi = score, i
        if bi is None: break
        sel.append(bi); rem.remove(bi)
    return [candidates[i] for i in sel]


# ── Lost-in-middle reorder ────────────────────────────────────────────────────
def reorder(candidates: list[dict]) -> list[dict]:
    if len(candidates) <= 2: return list(candidates)
    o = sorted(candidates, key=lambda x: x.get("ce_score", x.get("rrf_score", 0.0)), reverse=True)
    return [o[0], *o[2:], o[1]]


# ── Full retrieve_and_rerank ──────────────────────────────────────────────────
def retrieve_and_rerank(query: str) -> list[dict]:
    """Full retrieval pipeline — identical to Notebook B."""
    metadata_filter = extract_filters_from_query(query)
    namespaces      = route_namespaces(query)

    # multi-query fan-out
    subqueries  = _generate_subqueries(query)
    all_queries = [query] + subqueries
    all_lists: list[list[dict]] = []
    for q in all_queries:
        for ns in namespaces:
            filt = metadata_filter if ns == NS_TRANSACTIONS else None
            all_lists.append(_hybrid_query(q, alpha=1.0, top_k=TOP_K_RETRIEVAL, namespace=ns, metadata_filter=filt))
            all_lists.append(_hybrid_query(q, alpha=0.0, top_k=TOP_K_RETRIEVAL, namespace=ns, metadata_filter=filt))

    fused    = reciprocal_rank_fusion(all_lists, k=RRF_K)
    reranked = rerank_cross_encoder(query, fused[:TOP_K_RETRIEVAL], TOP_K_RERANK)
    diverse  = mmr_filter(reranked, query, TOP_K_MMR, MMR_LAMBDA)
    return reorder(diverse)[:TOP_K_FINAL]


def _generate_subqueries(query: str, n: int = N_SUBQUERIES) -> list[str]:
    _set_ollama_host(OLLAMA_BASE_URL)
    prompt = f"""Generate {n} alternative search queries for Singapore HDB property data.
Return ONLY a numbered list. No explanations.
Original: {query}
"""
    try:
        response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
        raw   = (response.get("message") or {}).get("content", "")
        lines = re.findall(r"^\s*\d+\.\s*(.+)$", raw, re.MULTILINE)
        return [l.strip().strip('"') for l in lines[:n]]
    except Exception:
        return []


# ── Answer generation ─────────────────────────────────────────────────────────
def generate_answer(query: str, context_chunks: list[dict]) -> str:
    """Generate grounded answer using Gemma 3."""
    _set_ollama_host(OLLAMA_BASE_URL)
    system = """You are a Singapore HDB property pricing assistant for PropertyLens.
Rules:
1. Answer ONLY using the provided context. No outside knowledge.
2. Cite every claim with [Context N] labels.
3. Give a Fair / Above market / Below market verdict ONLY for price fairness questions.
4. For amenity, trend, or explanation questions, do NOT give a price verdict.
5. If evidence is thin, say so. Keep answers to 3-5 sentences.
"""
    parts = ["## Retrieved context"]
    for i, c in enumerate(context_chunks, 1):
        md  = c.get("metadata") or {}
        txt = str(md.get("parent_text") or "").strip()
        parts.append(f"[Context {i}] source={md.get('source')} town={md.get('town')} year={md.get('sale_year')}")
        parts.append(txt)
        parts.append("")
    parts.extend(["## Question", query])
    user_prompt = "\n".join(parts).strip()
    try:
        response = ollama.chat(
            model=OLLAMA_MODEL,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user_prompt},
            ],
        )
        return (response.get("message") or {}).get("content", "")
    except Exception as e:
        return f"[Ollama error] {e}"


print("Pipeline functions loaded.")

### Evaluation test set

20 labelled query-answer pairs across all five source types.

Each entry has:
- `query` — the natural language question
- `persona` — Buyer / Seller / Trends / Amenities / XAI
- `expected_answer` — a short reference answer used by the faithfulness and relevance judges
- `relevant_keywords` — words that should appear in at least one retrieved chunk (used for Hit Rate)

**How to extend:** add more rows to `EVAL_SET`. The more entries you have the more reliable the aggregate scores.


In [ ]:
from __future__ import annotations

EVAL_SET: list[dict] = [
    # ── Buyer — price fairness ────────────────────────────────────────────────
    {
        "query":            "Is $580k fair for a 4-room HDB in Tampines?",
        "persona":          "Buyer",
        "expected_answer":  "Based on recent Tampines 4-room transactions the price is roughly fair or slightly above market depending on floor area and storey.",
        "relevant_keywords": ["TAMPINES", "4 ROOM"],
    },
    {
        "query":            "Is $430,000 fair for a 3-room HDB in Serangoon, 64 sqm, lease started 1978?",
        "persona":          "Buyer",
        "expected_answer":  "A 3-room Serangoon flat with 52 years remaining lease at $430k is close to market rates given recent comparable transactions.",
        "relevant_keywords": ["SERANGOON", "3 ROOM"],
    },
    {
        "query":            "Is $900k reasonable for a 5-room HDB in Queenstown?",
        "persona":          "Buyer",
        "expected_answer":  "Queenstown 5-room flats have transacted above $800k in recent years so $900k could be within range for premium units.",
        "relevant_keywords": ["QUEENSTOWN", "5 ROOM"],
    },
    {
        "query":            "How much should I offer for a 4-room flat in Bedok with asking price $620k?",
        "persona":          "Buyer",
        "expected_answer":  "Recent Bedok 4-room transactions suggest a counter-offer between $580k and $610k may be reasonable.",
        "relevant_keywords": ["BEDOK", "4 ROOM"],
    },
    # ── Seller — listing price ────────────────────────────────────────────────
    {
        "query":            "What should I list my 5-room Bishan flat for given current market trends?",
        "persona":          "Seller",
        "expected_answer":  "Bishan 5-room flats have been listing between $800k and $860k in recent transactions, suggesting a listing price around $820k–$860k.",
        "relevant_keywords": ["BISHAN", "5 ROOM"],
    },
    {
        "query":            "I want to sell my 3-room flat in Ang Mo Kio. What price should I set?",
        "persona":          "Seller",
        "expected_answer":  "Ang Mo Kio 3-room flats have varied widely by storey and lease remaining; check comparable transactions for your specific block.",
        "relevant_keywords": ["ANG MO KIO", "3 ROOM"],
    },
    # ── Trends ────────────────────────────────────────────────────────────────
    {
        "query":            "Are HDB prices in Queenstown rising or falling over the last 3 years?",
        "persona":          "Trends",
        "expected_answer":  "Queenstown median resale prices have been rising, from around $545k in 2019 to $690k in 2022.",
        "relevant_keywords": ["QUEENSTOWN", "trend", "median"],
    },
    {
        "query":            "Has the median price in Tampines increased since 2020?",
        "persona":          "Trends",
        "expected_answer":  "Yes, Tampines median resale prices have increased year-on-year since 2020 based on transaction data.",
        "relevant_keywords": ["TAMPINES", "trend", "median"],
    },
    {
        "query":            "Which year had the highest median resale price in Sengkang?",
        "persona":          "Trends",
        "expected_answer":  "Based on trend data, the most recent years show the highest median resale prices in Sengkang.",
        "relevant_keywords": ["SENGKANG", "median"],
    },
    # ── Amenities ─────────────────────────────────────────────────────────────
    {
        "query":            "What MRT stations and schools are near Bedok?",
        "persona":          "Amenities",
        "expected_answer":  "Bedok has two MRT stations (Bedok EW5, Bedok North DT29) and several primary schools including Temasek Primary.",
        "relevant_keywords": ["BEDOK", "MRT", "school"],
    },
    {
        "query":            "Are there shopping malls near Tampines HDB flats?",
        "persona":          "Amenities",
        "expected_answer":  "Tampines has multiple malls including Tampines Mall, Tampines 1, and Century Square.",
        "relevant_keywords": ["TAMPINES", "mall"],
    },
    {
        "query":            "How many hawker centres are in Ang Mo Kio?",
        "persona":          "Amenities",
        "expected_answer":  "Ang Mo Kio has 7 hawker centres including the Kebun Baru and Chong Boon markets.",
        "relevant_keywords": ["ANG MO KIO", "hawker"],
    },
    {
        "query":            "What amenities are near Serangoon HDB flats?",
        "persona":          "Amenities",
        "expected_answer":  "Serangoon has Serangoon MRT, several primary and secondary schools, and malls nearby.",
        "relevant_keywords": ["SERANGOON", "MRT"],
    },
    # ── XAI ───────────────────────────────────────────────────────────────────
    {
        "query":            "What features drive HDB resale prices the most according to the model?",
        "persona":          "XAI",
        "expected_answer":  "According to SHAP global importances, floor area, MRT proximity, and lease remaining years are top price drivers.",
        "relevant_keywords": ["SHAP", "feature", "floor_area"],
    },
    {
        "query":            "Why would a high-floor flat in Queenstown predict a higher price?",
        "persona":          "XAI",
        "expected_answer":  "High storey band positively influences price predictions — this is captured in SHAP drivers and comparable transactions.",
        "relevant_keywords": ["QUEENSTOWN", "storey"],
    },
    {
        "query":            "What pricing rules does the model use for Bishan flats?",
        "persona":          "XAI",
        "expected_answer":  "The surrogate rules suggest that Bishan flats with large floor area and remaining lease above 70 years typically price above median.",
        "relevant_keywords": ["rule", "BISHAN"],
    },
    # ── Cross-source ──────────────────────────────────────────────────────────
    {
        "query":            "Should I buy in Tampines or Bedok? Compare prices and amenities.",
        "persona":          "Cross-source",
        "expected_answer":  "Tampines generally has more malls and MRT options while Bedok has slightly lower median prices for similar flat types.",
        "relevant_keywords": ["TAMPINES", "BEDOK"],
    },
    {
        "query":            "Is a $650k asking price for a 5-room in Sengkang reasonable given recent trends?",
        "persona":          "Cross-source",
        "expected_answer":  "Recent Sengkang 5-room transactions and trend data should be checked to assess whether $650k is above or below market.",
        "relevant_keywords": ["SENGKANG", "5 ROOM"],
    },
    {
        "query":            "Are there good schools near affordable flats in Toa Payoh?",
        "persona":          "Cross-source",
        "expected_answer":  "Toa Payoh has several primary schools and MRT access; flat prices vary by storey and lease.",
        "relevant_keywords": ["TOA PAYOH", "school"],
    },
    {
        "query":            "What is a fair price for a 4-room flat in Woodlands with 70 years lease remaining?",
        "persona":          "Buyer",
        "expected_answer":  "Woodlands 4-room flats with 70+ years lease have transacted in a broad range — floor area and storey are key determinants.",
        "relevant_keywords": ["WOODLANDS", "4 ROOM"],
    },
]

print(f"Eval set: {len(EVAL_SET)} queries")
personas = {}
for e in EVAL_SET:
    personas[e['persona']] = personas.get(e['persona'], 0) + 1
for p, n in sorted(personas.items()):
    print(f"  {p:<22s}: {n}")

### Metric 1 — Hit Rate @ 5

**What it measures:** for each query, does at least one of the top-5 retrieved chunks contain a keyword that signals a relevant result?

**Why it's the most important metric here:** it directly evaluates the retrieval stack — hybrid BM25+cosine, weighted RRF, cross-encoder, and MMR. A low hit rate means no amount of LLM quality will save the answers.

**How it works:** each eval entry has `relevant_keywords`. A chunk is considered a hit if the chunk's `parent_text` or metadata contains at least one of those keywords. Hit Rate = fraction of queries where at least one chunk is a hit.

**Also computed:** Mean Reciprocal Rank (MRR) — the average of 1/rank_of_first_hit. MRR penalises cases where the relevant chunk only appears at slot 4 or 5.


In [ ]:
from __future__ import annotations
from IPython.display import Markdown, display
import html


def chunk_is_hit(chunk: dict, keywords: list[str]) -> bool:
    """
    Return True if any keyword appears in the chunk's parent_text or metadata.

    Case-insensitive match.

    Args:
        chunk: Pinecone match dict with metadata.
        keywords: list of strings to search for.

    Returns:
        True if any keyword found.
    """
    md       = chunk.get("metadata") or {}
    haystack = " ".join([
        str(md.get("parent_text") or ""),
        str(md.get("town") or ""),
        str(md.get("flat_type") or ""),
        str(md.get("amenity_type") or ""),
        str(md.get("xai_type") or ""),
    ]).upper()
    return any(kw.upper() in haystack for kw in keywords)


def evaluate_hit_rate(eval_set: list[dict]) -> list[dict]:
    """
    Run retrieval for every eval entry and compute Hit Rate and MRR.

    Args:
        eval_set: list of eval dicts with query + relevant_keywords.

    Returns:
        List of result dicts with hit (bool), rank (int|None), mrr (float).
    """
    results: list[dict] = []

    for i, entry in enumerate(eval_set, 1):
        query    = entry["query"]
        keywords = entry["relevant_keywords"]
        print(f"  [{i:02d}/{len(eval_set)}] {query[:60]}...")

        try:
            chunks = retrieve_and_rerank(query)
        except Exception as e:
            print(f"         ✗ retrieval failed: {e}")
            results.append({
                "query": query, "persona": entry["persona"],
                "hit": False, "rank": None, "mrr": 0.0,
                "chunks": [], "error": str(e),
            })
            continue

        hit_rank = None
        for rank, chunk in enumerate(chunks, 1):
            if chunk_is_hit(chunk, keywords):
                hit_rank = rank
                break

        hit = hit_rank is not None
        mrr = (1.0 / hit_rank) if hit_rank else 0.0
        symbol = "✓" if hit else "✗"
        print(f"         {symbol} hit={hit} rank={hit_rank} mrr={mrr:.2f}")

        results.append({
            "query":   query,
            "persona": entry["persona"],
            "hit":     hit,
            "rank":    hit_rank,
            "mrr":     mrr,
            "chunks":  chunks,
            "error":   None,
        })

    return results


print("Running Hit Rate evaluation...")
mem("before hit rate eval")
retrieval_results = evaluate_hit_rate(EVAL_SET)
mem("after hit rate eval")

hit_rate = sum(r["hit"] for r in retrieval_results) / len(retrieval_results)
mrr      = sum(r["mrr"] for r in retrieval_results) / len(retrieval_results)
print(f"\n{'='*40}")
print(f"Hit Rate @ 5 : {hit_rate:.2%}  ({sum(r['hit'] for r in retrieval_results)}/{len(retrieval_results)} queries)")
print(f"MRR          : {mrr:.3f}")
print(f"{'='*40}")

### Metric 2 — Faithfulness

**What it measures:** are the claims in the generated answer actually supported by the retrieved context, or did Gemma hallucinate?

**Why it matters most for PropertyLens:** this is a financial decision tool. A hallucinated price (e.g. Gemma inventing $580k is fair when all context shows $450k–$500k) could directly mislead buyers and sellers.

**How it works:** we use Gemma 3 as the judge. The judge prompt asks: "Given this context and this answer, are all factual claims in the answer supported by the context? Score 1 (fully grounded) to 5 (significant hallucination)." We invert the scale so 5 = fully faithful.

**Note:** using the same LLM as judge and generator is a known limitation (self-serving bias). For production, swap `judge_model` to a different model or use human annotators on a sample.


In [ ]:
from __future__ import annotations
import re
import ollama


def judge_faithfulness(
    query: str,
    context_chunks: list[dict],
    answer: str,
    judge_model: str = OLLAMA_MODEL,
) -> tuple[float, str]:
    """
    Use Gemma 3 as an LLM judge to score answer faithfulness.

    Asks: are all factual claims in the answer supported by the context?

    Args:
        query: original user question.
        context_chunks: retrieved chunks passed to the generator.
        answer: generated answer to evaluate.
        judge_model: Ollama model to use as judge.

    Returns:
        (score_1_to_5, reasoning_string)
        5 = fully grounded in context, 1 = significant hallucination.
    """
    _set_ollama_host(OLLAMA_BASE_URL)

    ctx_text = "\n".join(
        f"[Context {i}] {str((c.get('metadata') or {}).get('parent_text', ''))[:400]}"
        for i, c in enumerate(context_chunks, 1)
    )

    prompt = f"""You are evaluating a RAG system for Singapore HDB property pricing.

QUESTION: {query}

RETRIEVED CONTEXT:
{ctx_text}

GENERATED ANSWER:
{answer}

Task: Score whether every factual claim in the answer is supported by the retrieved context.
Ignore style and completeness — only check if claims contradict or go beyond the context.

Score 1-5:
5 = every claim is directly supported by context
4 = mostly grounded, one minor unsupported detail
3 = mix of grounded and unsupported claims
2 = significant claims not in context
1 = answer mostly ignores or contradicts the context

Return ONLY this format:
SCORE: <number 1-5>
REASON: <one sentence>
"""
    try:
        response = ollama.chat(
            model=judge_model,
            messages=[{"role": "user", "content": prompt}],
        )
        raw    = (response.get("message") or {}).get("content", "")
        score_m  = re.search(r"SCORE:\s*([1-5])", raw)
        reason_m = re.search(r"REASON:\s*(.+)", raw)
        score  = float(score_m.group(1)) if score_m else 3.0
        reason = reason_m.group(1).strip() if reason_m else raw[:120]
        return score, reason
    except Exception as e:
        return 3.0, f"judge failed: {e}"


def evaluate_faithfulness(
    eval_set: list[dict],
    retrieval_results: list[dict],
) -> list[dict]:
    """
    Generate answers for each eval entry and score faithfulness.

    Args:
        eval_set: eval entries.
        retrieval_results: output of evaluate_hit_rate() — reuses retrieved chunks.

    Returns:
        List of result dicts with answer, faith_score, faith_reason.
    """
    results: list[dict] = []

    for i, (entry, ret) in enumerate(zip(eval_set, retrieval_results), 1):
        query  = entry["query"]
        chunks = ret.get("chunks", [])
        print(f"  [{i:02d}/{len(eval_set)}] {query[:55]}...")

        if not chunks:
            results.append({
                "query":        query,
                "persona":      entry["persona"],
                "answer":       "",
                "faith_score":  1.0,
                "faith_reason": "no chunks retrieved",
            })
            print("         ✗ no chunks — skipping")
            continue

        try:
            answer = generate_answer(query, chunks)
            score, reason = judge_faithfulness(query, chunks, answer)
            print(f"         faithfulness={score:.1f} | {reason[:80]}")
            results.append({
                "query":        query,
                "persona":      entry["persona"],
                "answer":       answer,
                "faith_score":  score,
                "faith_reason": reason,
            })
        except Exception as e:
            print(f"         ✗ failed: {e}")
            results.append({
                "query":        query,
                "persona":      entry["persona"],
                "answer":       "",
                "faith_score":  1.0,
                "faith_reason": str(e),
            })

    return results


print("Running Faithfulness evaluation...")
mem("before faithfulness eval")
faithfulness_results = evaluate_faithfulness(EVAL_SET, retrieval_results)
mem("after faithfulness eval")

mean_faith = sum(r["faith_score"] for r in faithfulness_results) / len(faithfulness_results)
print(f"\n{'='*40}")
print(f"Mean Faithfulness : {mean_faith:.2f} / 5.0")
print(f"{'='*40}")

### Metric 3 — Answer relevance

**What it measures:** does the generated answer actually address the question that was asked?

**Why it matters:** even with perfect retrieval and full faithfulness, Gemma could give a technically grounded but unhelpful answer (e.g. describing amenities when asked about price). This catches that.

**How it works:** Gemma 3 as judge again. The judge compares the answer to the `expected_answer` reference and scores relevance 1–5. 5 = fully addresses the question, 1 = answer is off-topic or a non-answer.


In [ ]:
from __future__ import annotations
import re
import ollama


def judge_answer_relevance(
    query: str,
    expected_answer: str,
    actual_answer: str,
    judge_model: str = OLLAMA_MODEL,
) -> tuple[float, str]:
    """
    Use Gemma 3 to score whether the actual answer addresses the question.

    Compares against expected_answer as a reference direction (not exact match).

    Args:
        query: original question.
        expected_answer: reference direction from EVAL_SET.
        actual_answer: answer generated by the RAG pipeline.
        judge_model: Ollama model to use.

    Returns:
        (score_1_to_5, reasoning_string)
        5 = directly and fully answers the question, 1 = off-topic.
    """
    _set_ollama_host(OLLAMA_BASE_URL)

    prompt = f"""You are evaluating a Singapore HDB property Q&A system.

QUESTION: {query}

REFERENCE ANSWER (direction, not exact):
{expected_answer}

ACTUAL ANSWER:
{actual_answer}

Task: Score whether the actual answer addresses the question.
Use the reference only as a guide to the expected direction — do not penalise different wording.
Penalise: non-answers, off-topic responses, refusing to answer when evidence exists, wrong verdict.

Score 1-5:
5 = directly and completely answers the question
4 = mostly answers with minor gaps
3 = partially answers, some important aspect missing
2 = tangentially related but does not answer
1 = does not answer the question

Return ONLY this format:
SCORE: <number 1-5>
REASON: <one sentence>
"""
    try:
        response = ollama.chat(
            model=judge_model,
            messages=[{"role": "user", "content": prompt}],
        )
        raw      = (response.get("message") or {}).get("content", "")
        score_m  = re.search(r"SCORE:\s*([1-5])", raw)
        reason_m = re.search(r"REASON:\s*(.+)", raw)
        score    = float(score_m.group(1)) if score_m else 3.0
        reason   = reason_m.group(1).strip() if reason_m else raw[:120]
        return score, reason
    except Exception as e:
        return 3.0, f"judge failed: {e}"


def evaluate_answer_relevance(
    eval_set: list[dict],
    faithfulness_results: list[dict],
) -> list[dict]:
    """
    Score answer relevance for each eval entry.

    Reuses answers already generated in faithfulness evaluation.

    Args:
        eval_set: eval entries with expected_answer.
        faithfulness_results: output of evaluate_faithfulness() — reuses answers.

    Returns:
        List of result dicts with rel_score, rel_reason.
    """
    results: list[dict] = []

    for i, (entry, faith) in enumerate(zip(eval_set, faithfulness_results), 1):
        query    = entry["query"]
        expected = entry["expected_answer"]
        actual   = faith.get("answer", "")
        print(f"  [{i:02d}/{len(eval_set)}] {query[:55]}...")

        if not actual:
            results.append({
                "query":      query,
                "persona":    entry["persona"],
                "rel_score":  1.0,
                "rel_reason": "no answer generated",
            })
            print("         ✗ no answer — skipping")
            continue

        try:
            score, reason = judge_answer_relevance(query, expected, actual)
            print(f"         relevance={score:.1f} | {reason[:80]}")
            results.append({
                "query":      query,
                "persona":    entry["persona"],
                "rel_score":  score,
                "rel_reason": reason,
            })
        except Exception as e:
            print(f"         ✗ failed: {e}")
            results.append({
                "query":      query,
                "persona":    entry["persona"],
                "rel_score":  1.0,
                "rel_reason": str(e),
            })

    return results


print("Running Answer Relevance evaluation...")
mem("before relevance eval")
relevance_results = evaluate_answer_relevance(EVAL_SET, faithfulness_results)
mem("after relevance eval")

mean_rel = sum(r["rel_score"] for r in relevance_results) / len(relevance_results)
print(f"\n{'='*40}")
print(f"Mean Answer Relevance : {mean_rel:.2f} / 5.0")
print(f"{'='*40}")

### Summary table

All three metrics per query, plus aggregate means and per-persona breakdowns.
Use this table to identify which query types need the most improvement.


In [ ]:
from __future__ import annotations
import pandas as pd
from IPython.display import display


def build_summary_table(
    eval_set: list[dict],
    retrieval_results: list[dict],
    faithfulness_results: list[dict],
    relevance_results: list[dict],
) -> pd.DataFrame:
    """
    Combine all three metric results into one summary DataFrame.

    Returns:
        DataFrame with columns: persona, query, hit, rank, mrr,
        faithfulness, answer_relevance.
    """
    rows = []
    for entry, ret, faith, rel in zip(
        eval_set, retrieval_results, faithfulness_results, relevance_results
    ):
        rows.append({
            "persona":          entry["persona"],
            "query":            entry["query"][:60] + "..." if len(entry["query"]) > 60 else entry["query"],
            "hit@5":            "✓" if ret.get("hit") else "✗",
            "rank":             ret.get("rank") or "-",
            "mrr":              f"{ret.get('mrr', 0.0):.2f}",
            "faithfulness":     f"{faith.get('faith_score', 0.0):.1f}/5",
            "answer_relevance": f"{rel.get('rel_score', 0.0):.1f}/5",
        })
    return pd.DataFrame(rows)


df = build_summary_table(
    EVAL_SET, retrieval_results, faithfulness_results, relevance_results
)

print("\n" + "="*70)
print("PER-QUERY RESULTS")
print("="*70)
display(df.to_string(index=False))

# ── aggregate metrics ─────────────────────────────────────────────────────────
n          = len(EVAL_SET)
hit_rate   = sum(r.get("hit", False) for r in retrieval_results) / n
mean_mrr   = sum(r.get("mrr", 0.0) for r in retrieval_results) / n
mean_faith = sum(r.get("faith_score", 0.0) for r in faithfulness_results) / n
mean_rel   = sum(r.get("rel_score", 0.0) for r in relevance_results) / n

print("\n" + "="*40)
print("AGGREGATE SCORES")
print("="*40)
print(f"  Hit Rate @ 5       : {hit_rate:.1%}")
print(f"  MRR                : {mean_mrr:.3f}")
print(f"  Faithfulness       : {mean_faith:.2f} / 5.0")
print(f"  Answer Relevance   : {mean_rel:.2f} / 5.0")
print("="*40)

# ── per-persona breakdown ─────────────────────────────────────────────────────
print("\nPER-PERSONA BREAKDOWN")
print("-"*60)
persona_groups: dict[str, list] = {}
for entry, ret, faith, rel in zip(
    EVAL_SET, retrieval_results, faithfulness_results, relevance_results
):
    p = entry["persona"]
    if p not in persona_groups:
        persona_groups[p] = []
    persona_groups[p].append({
        "hit":   ret.get("hit", False),
        "mrr":   ret.get("mrr", 0.0),
        "faith": faith.get("faith_score", 0.0),
        "rel":   rel.get("rel_score", 0.0),
    })

for persona, rows in sorted(persona_groups.items()):
    n_p     = len(rows)
    hr      = sum(r["hit"] for r in rows) / n_p
    mrr_p   = sum(r["mrr"] for r in rows) / n_p
    faith_p = sum(r["faith"] for r in rows) / n_p
    rel_p   = sum(r["rel"] for r in rows) / n_p
    print(f"  {persona:<22s} hit={hr:.0%} mrr={mrr_p:.2f} faith={faith_p:.1f} rel={rel_p:.1f}  (n={n_p})")

print()
mem("eval complete")

### How to interpret the scores

#### Hit Rate @ 5
| Score | Interpretation | Action |
|-------|---------------|--------|
| >80% | Good retrieval | Move on to generation quality |
| 60–80% | Acceptable for dev, improve before prod | Tune `SOURCE_WEIGHTS`, increase `SAMPLE_SIZE` to full corpus |
| <60% | Retrieval is broken | Check BM25 cache is from correct corpus, check namespace routing |

#### Faithfulness
| Score | Interpretation | Action |
|-------|---------------|--------|
| 4.0–5.0 | Gemma stays grounded | Acceptable for production |
| 3.0–4.0 | Occasional unsupported claims | Tighten system prompt Rule 1, add "do not speculate" |
| <3.0 | Significant hallucination | Add explicit "quote only" instruction, increase context size |

#### Answer Relevance
| Score | Interpretation | Action |
|-------|---------------|--------|
| 4.0–5.0 | Gemma addresses questions well | Good |
| 3.0–4.0 | Some off-topic answers | Check namespace routing — are trend queries getting trend chunks? |
| <3.0 | Gemma frequently dodges or misses | Improve system prompt, add few-shot examples |

#### Known limitations of this evaluation
- **LLM-as-judge bias**: using the same Gemma 3 model as both generator and judge introduces self-serving bias — the judge may score its own outputs more generously.
- **Small test set**: 20 queries is enough to identify obvious failure modes but not for statistically reliable comparisons between configurations.
- **Keyword-based hit detection**: the Hit Rate check uses keyword matching which can give false positives. A chunk with "TAMPINES" in the amenity summary is counted as a hit even for transaction queries.
- **No ground truth prices**: the `expected_answer` references are directional, not exact. The judge cannot verify a specific price claim is correct.

#### Suggested next steps
- Increase test set to 50+ queries for more reliable aggregate scores.
- Run with a different judge model (e.g. `llama3`) to reduce self-serving bias.
- Add NDCG@5 by manually labelling chunk relevance (0 = not relevant, 1 = relevant, 2 = highly relevant) for a subset of queries.
- Compare scores before/after changing `SOURCE_WEIGHTS` or `TOP_K_RERANK` to measure the effect of pipeline changes.
